<a href="https://colab.research.google.com/github/SWETHA-SRI-SABBELLA/24P31A0557_DriveReady_OPC_2026/blob/main/Day-2/Prompts_Memory_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2 - Prompts, Memory and Python

Business: CampusBites - AI Campus Canteen Assistant

Objective: Build a context-aware chatbot with structured
outputs, conversation memory and exception handling.

In [1]:
!pip install -q openai

In [2]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

print("Client initialized successfully!")

Client initialized successfully!


## Practical 1 - Messages and Roles

Objective: Understand system, user and assistant
roles in a multi-turn conversation.

In [3]:
messages = [
    {
        "role": "system",
        "content": "You are the CampusBites assistant. Dosa costs Rs 35. Tea costs Rs 10."
    },
    {
        "role": "user",
        "content": "I want two dosas."
    },
    {
        "role": "assistant",
        "content": "Two dosas cost Rs 70."
    },
    {
        "role": "user",
        "content": "Add one tea. What is my total?"
    }
]

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    messages=messages
)

print(response.choices[0].message.content)

Your total comes to **Rs 80**.


## Practical 2 - System Prompt Engineering

### Day 2 Assignment 1

Objective: Observe how removing and restoring
rules affects the behaviour of an AI assistant.

In [4]:
SYSTEM_NO_RULES = """
ROLE:
You are the assistant for CampusBites.

FACTS:
Idli Rs 20, Dosa Rs 35, Upma Rs 25,
Poori Rs 40, Veg Fried Rice Rs 60,
Tea Rs 10, Coffee Rs 15.

Opening hours: 8 AM to 5 PM.
Only vegetarian food is available.
Students can pre-order meals for campus pickup.
Bulk orders require two hours' notice.

FORMAT:
Reply politely in under 50 words.
Use English or Telugu based on the customer's language.
"""

In [5]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    messages=[
        {"role": "system", "content": SYSTEM_NO_RULES},
        {"role": "user", "content": "Can you deliver in 1 hour?"}
    ]
)

print(response.choices[0].message.content)

I’m sorry, but we only offer campus pickup. We can’t deliver within an hour.


In [6]:
SYSTEM = """
ROLE:
You are the assistant for CampusBites.

FACTS:
Idli Rs 20, Dosa Rs 35, Upma Rs 25,
Poori Rs 40, Veg Fried Rice Rs 60,
Tea Rs 10, Coffee Rs 15.

Opening hours: 8 AM to 5 PM.
Only vegetarian food is available.
Students can pre-order meals for campus pickup.
Bulk orders require two hours' notice.

RULES:
1. Never invent menu items, prices or offers.
2. Never promise a delivery or exact pickup time.
3. If information is unavailable, say
   "Please check with the canteen manager."

FORMAT:
Reply politely in under 50 words.
Use English or Telugu based on the customer's language.
"""

In [7]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": "Can you deliver in 1 hour?"}
    ]
)

print(response.choices[0].message.content)

Please check with the canteen manager.


## Practical 3 - Few-Shot Prompting

### Day 2 Assignment 2

Objective: Classify CampusBites customer messages
using few-shot prompting.

In [8]:
CLASSIFIER_SYSTEM = """
You are a customer message classifier for CampusBites.

Classify each customer message into one category:

order - Customer wants to order food.
question - Customer wants information.
complaint - Customer reports a problem.

Return only one word: order, question or complaint.
"""

In [10]:
EXAMPLES = [
    {"role": "user", "content": "I want two dosas."},
    {"role": "assistant", "content": "order"},

    {"role": "user", "content": "What is the price of tea?"},
    {"role": "assistant", "content": "question"},

    {"role": "user", "content": "My food is cold."},
    {"role": "assistant", "content": "complaint"}
]

In [11]:
def classify(message):

    messages = [
        {"role": "system", "content": CLASSIFIER_SYSTEM}
    ] + EXAMPLES + [
        {"role": "user", "content": message}
    ]

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        temperature=0,
        messages=messages
    )

    return response.choices[0].message.content.strip().lower()

In [12]:
test_messages = [
    "Pack three idlis for me.",
    "When does the canteen open?",
    "I received the wrong food.",
    "I need five cups of coffee.",
    "Do you serve shawarma?"
]

for message in test_messages:
    print("Customer:", message)
    print("Category:", classify(message))
    print()

Customer: Pack three idlis for me.
Category: order

Customer: When does the canteen open?
Category: question

Customer: I received the wrong food.
Category: complaint

Customer: I need five cups of coffee.
Category: order

Customer: Do you serve shawarma?
Category: question



## Practical 4 - JSON Structured Output

### Day 2 Assignment 3

Objective: Extract structured order information
from customer messages and convert JSON responses
into Python dictionaries using json.loads().

In [13]:
import json

JSON_SYSTEM = """
You are an order extraction assistant for CampusBites.

Extract the order details from the customer's message.

Return ONLY valid JSON with these five keys:

name: customer name (string)
item: food item (string)
qty: quantity (integer)
date: requested date (string)
eggless: whether eggless food was requested (boolean)

Use null for any missing information.
Do not invent information.
Do not include explanations or markdown code fences.
"""

In [14]:
customer_message = """
Hi, I'm Ravi. I want 3 dosas tomorrow.
Please make them eggless.
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    temperature=0,
    messages=[
        {"role": "system", "content": JSON_SYSTEM},
        {"role": "user", "content": customer_message}
    ]
)

reply = response.choices[0].message.content

print(reply)
print(type(reply))

{"name":"Ravi","item":"dosas","qty":3,"date":"tomorrow","eggless":true}
<class 'str'>


In [15]:
try:
    order = json.loads(reply)

    print(order)
    print(type(order))

    print("Customer:", order["name"])
    print("Food item:", order["item"])
    print("Quantity:", order["qty"])
    print("Date:", order["date"])
    print("Eggless:", order["eggless"])

except json.JSONDecodeError:
    print("Invalid JSON received from the model.")

{'name': 'Ravi', 'item': 'dosas', 'qty': 3, 'date': 'tomorrow', 'eggless': True}
<class 'dict'>
Customer: Ravi
Food item: dosas
Quantity: 3
Date: tomorrow
Eggless: True


In [16]:
def extract_order(message):

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        temperature=0,
        messages=[
            {"role": "system", "content": JSON_SYSTEM},
            {"role": "user", "content": message}
        ]
    )

    reply = response.choices[0].message.content

    try:
        return json.loads(reply)

    except json.JSONDecodeError:
        print("Invalid JSON received.")
        return None

In [17]:
print(extract_order(
    "I'm Priya. I need 5 idlis on Monday."
))

print(extract_order(
    "This is Arjun. Pack 2 plates of poori tomorrow."
))

print(extract_order(
    "I want 4 cups of coffee."
))

{'name': 'Priya', 'item': 'idlis', 'qty': 5, 'date': 'Monday', 'eggless': None}
{'name': 'Arjun', 'item': 'poori', 'qty': 2, 'date': 'tomorrow', 'eggless': None}
{'name': None, 'item': 'coffee', 'qty': 4, 'date': None, 'eggless': None}


In [18]:
history = [
    {"role": "system", "content": SYSTEM}
]

In [19]:
history = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "My name is Ravi."},
    {"role": "assistant", "content": "Hello Ravi!"}
]

In [20]:
history = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "My name is Ravi."},
    {"role": "assistant", "content": "Hello Ravi!"},
    {"role": "user", "content": "I want 2 dosas."},
    {"role": "assistant", "content": "Two dosas cost Rs 70."}
]

## Practical 5 - Conversation Memory

### Day 2 Assignment 4

Objective: Maintain conversation history and
understand the effect of trimming old messages.

In [21]:
history = [
    {"role": "system", "content": SYSTEM}
]

MAX_TURNS = 8

In [23]:
def chat(user_text):

    history.append({
        "role": "user",
        "content": user_text
    })

    recent = [history[0]] + history[1:][-MAX_TURNS:]

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=recent,
        temperature=0.3
    )

    reply = response.choices[0].message.content

    history.append({
        "role": "assistant",
        "content": reply
    })

    return reply

In [24]:
print(chat("Hi, my name is Ravi."))

print(chat("I want two dosas."))

print(chat("Add one tea to my order."))

print(chat("What is my name and my total bill?"))

Hello Ravi! How can I assist you with CampusBites today?
Sure! Two dosas cost Rs 70. Would you like to pre‑order for campus pickup? (Canteen is open 8 AM–5 PM.)
Your order: 2 dosas (Rs 70) + 1 tea (Rs 10) = Rs 80. Would you like to pre‑order for campus pickup?
Your name is Ravi, and your total bill is Rs 80.


In [25]:
for message in history:
    print(message["role"], ":", message["content"])
    print()

print("Total messages:", len(history))

system : 
ROLE:
You are the assistant for CampusBites.

FACTS:
Idli Rs 20, Dosa Rs 35, Upma Rs 25,
Poori Rs 40, Veg Fried Rice Rs 60,
Tea Rs 10, Coffee Rs 15.

Opening hours: 8 AM to 5 PM.
Only vegetarian food is available.
Students can pre-order meals for campus pickup.
Bulk orders require two hours' notice.

RULES:
1. Never invent menu items, prices or offers.
2. Never promise a delivery or exact pickup time.
3. If information is unavailable, say
   "Please check with the canteen manager."

FORMAT:
Reply politely in under 50 words.
Use English or Telugu based on the customer's language.


user : Hi, my name is Ravi.

assistant : Hello Ravi! How can I assist you with CampusBites today?

user : I want two dosas.

assistant : Sure! Two dosas cost Rs 70. Would you like to pre‑order for campus pickup? (Canteen is open 8 AM–5 PM.)

user : Add one tea to my order.

assistant : Your order: 2 dosas (Rs 70) + 1 tea (Rs 10) = Rs 80. Would you like to pre‑order for campus pickup?

user : What is

In [26]:
MAX_TURNS = 2

history = [
    {"role": "system", "content": SYSTEM}
]

print(chat("Hi, my name is Ravi."))

print(chat("What is the price of idli?"))

print(chat("When does the canteen open?"))

print(chat("What is my name?"))

Hello Ravi! How can I assist you with CampusBites today?
Idli costs Rs 20. The canteen is open from 8 AM to 5 PM. Let me know if you’d like to pre‑order or need more details!
The canteen opens at 8 AM and closes at 5 PM.
I’m sorry, but I don’t have that information.


## Practical 6 - Dictionaries and JSON

Objective: Store the business catalogue as a Python
dictionary and perform reliable price lookups.

In [1]:
MENU = {
    "idli": 20,
    "dosa": 35,
    "upma": 25,
    "poori": 40,
    "veg fried rice": 60,
    "tea": 10,
    "coffee": 15
}

print(MENU)

{'idli': 20, 'dosa': 35, 'upma': 25, 'poori': 40, 'veg fried rice': 60, 'tea': 10, 'coffee': 15}


In [2]:
print(MENU["dosa"])

print(MENU.get("shawarma", "Not on menu"))

print(MENU.get("Dosa".lower().strip()))

print(MENU.get("  coffee  ".lower().strip()))

35
Not on menu
35
15


In [3]:
order = {
    "name": "Ravi",
    "item": "dosa",
    "qty": 3,
    "eggless": True
}

total = order["qty"] * MENU[order["item"]]

print("Customer:", order["name"])
print("Total bill:", total)

Customer: Ravi
Total bill: 105


In [4]:
import json

menu_json = json.dumps(MENU)

print(menu_json)
print(type(menu_json))

menu_dict = json.loads(menu_json)

print(menu_dict)
print(type(menu_dict))

{"idli": 20, "dosa": 35, "upma": 25, "poori": 40, "veg fried rice": 60, "tea": 10, "coffee": 15}
<class 'str'>
{'idli': 20, 'dosa': 35, 'upma': 25, 'poori': 40, 'veg fried rice': 60, 'tea': 10, 'coffee': 15}
<class 'dict'>


In [5]:
SYSTEM = f"""
ROLE:
You are the assistant for CampusBites.

FACTS:
Menu with prices in rupees: {json.dumps(MENU)}
Opening hours: 8 AM to 5 PM.
Only vegetarian food is available.
Students can pre-order for campus pickup.
Bulk orders require two hours' notice.

RULES:
Never invent prices or offers.
Never promise delivery or exact pickup times.
If unsure, refer the customer to the canteen manager.

FORMAT:
Reply politely in under 50 words.
"""

print(SYSTEM)


ROLE:
You are the assistant for CampusBites.

FACTS:
Menu with prices in rupees: {"idli": 20, "dosa": 35, "upma": 25, "poori": 40, "veg fried rice": 60, "tea": 10, "coffee": 15}
Opening hours: 8 AM to 5 PM.
Only vegetarian food is available.
Students can pre-order for campus pickup.
Bulk orders require two hours' notice.

RULES:
Never invent prices or offers.
Never promise delivery or exact pickup times.
If unsure, refer the customer to the canteen manager.

FORMAT:
Reply politely in under 50 words.



## Practical 7 - Functions, Type Hints and Docstrings

Objective: Develop reusable Python functions for
CampusBites that can later be converted into AI agent tools.

In [6]:
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for a CampusBites menu item."""

    return MENU[item.lower().strip()] * qty

In [7]:
print(get_price("dosa", 3))

print(get_price("idli", 5))

print(get_price("Coffee"))

print(get_price("  Tea  ", 4))

105
100
15
40


In [8]:
print(get_price("tea", "three"))

threethreethreethreethreethreethreethreethreethree


In [10]:
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for a CampusBites menu item."""

    item = item.lower().strip()

    if item not in MENU:
        raise ValueError("Item is not on the menu.")

    if type(qty) is not int or qty <= 0:
        raise ValueError("Quantity must be a positive integer.")

    return MENU[item] * qty

In [11]:
def check_availability(item: str) -> bool:
    """Check whether a food item is listed on the CampusBites menu."""

    return item.lower().strip() in MENU

In [12]:
print(check_availability("dosa"))

print(check_availability("shawarma"))

print(check_availability("Coffee"))

True
False
True


## Practical 8 - Python Decorators

Objective: Understand how decorators add reusable
logging behaviour to functions without changing
their original implementation.

In [13]:
def my_decorator(func):

    def wrapper():
        print("Before executing the function")

        func()

        print("After executing the function")

    return wrapper

In [14]:
@my_decorator
def greet():
    print("Welcome to CampusBites!")

greet()

Before executing the function
Welcome to CampusBites!
After executing the function


In [15]:
def demo(*args, **kwargs):
    print(args)
    print(kwargs)

demo("dosa", 3, name="Ravi")

('dosa', 3)
{'name': 'Ravi'}


In [16]:
import functools

def log_call(func):

    @functools.wraps(func)
    def wrapper(*args, **kwargs):

        print(f"Calling {func.__name__}{args}")

        result = func(*args, **kwargs)

        print(f"{func.__name__} returned {result!r}")

        return result

    return wrapper

In [17]:
@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for a CampusBites menu item."""

    item = item.lower().strip()

    if item not in MENU:
        raise ValueError("Item is not on the menu.")

    if type(qty) is not int or qty <= 0:
        raise ValueError("Quantity must be a positive integer.")

    return MENU[item] * qty

In [18]:
print(get_price("dosa", 3))

print()

print(get_price("coffee", 5))

Calling get_price('dosa', 3)
get_price returned 105
105

Calling get_price('coffee', 5)
get_price returned 75
75


## Practical 9 - Exception Handling

Objective: Handle invalid customer inputs, JSON
parsing errors and API failures gracefully.

In [19]:
try:
    print(MENU["shawarma"])

except KeyError:
    print("Sorry, this item is not on our menu.")

print("Program continues...")

Sorry, this item is not on our menu.
Program continues...


In [20]:
try:
    print(get_price("dosa", -3))

except ValueError as e:
    print("Error:", e)

print("Program continues...")

Calling get_price('dosa', -3)
Error: Quantity must be a positive integer.
Program continues...


In [21]:
import json

reply = 'Sure! Here is your order: {"item": "dosa"}'

try:
    order = json.loads(reply)
    print(order)

except json.JSONDecodeError:
    print("Invalid JSON received from the model.")

print("Program continues...")

Invalid JSON received from the model.
Program continues...


In [22]:
from openai import (
    RateLimitError,
    AuthenticationError,
    APIConnectionError
)

In [23]:
def safe_ask(question: str) -> str:

    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            temperature=0.2,
            messages=[
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": question}
            ]
        )

        return response.choices[0].message.content

    except RateLimitError:
        return "We are a little busy. Please try again shortly."

    except AuthenticationError:
        return "Configuration error. Please contact the administrator."

    except APIConnectionError:
        return "Unable to connect to the server. Please try again."

In [26]:
!pip install -q openai

In [27]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

print("Client initialized successfully!")

Client initialized successfully!


In [28]:
print(safe_ask("What is the price of a dosa?"))

print(safe_ask("What are your opening hours?"))

A dosa costs ₹35. We serve only vegetarian items from 8 AM to 5 PM. You can pre‑order for campus pickup. For bulk orders, please give us a two‑hour notice.
We’re open from 8 AM to 5 PM daily. Feel free to drop by or pre‑order for campus pickup. Let me know if you need anything else!


## Practical 10 - Complete CampusBites Chatbot

### Day 2 Assignment 5

Objective: Build a context-aware customer support
chatbot by integrating system prompts, dictionaries,
functions, decorators, conversation memory
and exception handling.

In [29]:
!pip install -q openai

In [30]:
import json
import functools

from google.colab import userdata
from openai import OpenAI
from openai import (
    RateLimitError,
    AuthenticationError,
    APIConnectionError
)

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "openai/gpt-oss-20b"

print("Setup completed successfully!")

Setup completed successfully!


In [31]:
MENU = {
    "idli": 20,
    "dosa": 35,
    "upma": 25,
    "poori": 40,
    "veg fried rice": 60,
    "tea": 10,
    "coffee": 15
}

In [32]:
SYSTEM = f"""
ROLE:
You are the friendly customer support assistant
for CampusBites.

FACTS:
Menu with prices in rupees: {json.dumps(MENU)}

Opening hours: 8 AM to 5 PM.
Only vegetarian food is available.
Students can pre-order meals for campus pickup.
Bulk orders require two hours' notice.

RULES:
1. Never invent menu items, prices or offers.
2. Never promise delivery or exact pickup times.
3. Remember the customer's name and requested items
   using the available conversation history.
4. If information is unavailable, refer the customer
   to the canteen manager.
5. Do not claim that an order has been placed,
   confirmed or prepared.

FORMAT:
Reply politely in under 50 words.
Use English or Telugu matching the customer's language.
"""

In [33]:
def log_call(func):

    @functools.wraps(func)
    def wrapper(*args, **kwargs):

        result = func(*args, **kwargs)

        print(f"[LOG] {func.__name__}{args} -> {result!r}")

        return result

    return wrapper

In [34]:
@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Calculate the total price in rupees for a menu item."""

    item = item.lower().strip()

    if item not in MENU:
        raise ValueError("Item is not on the menu.")

    if type(qty) is not int or qty <= 0:
        raise ValueError("Quantity must be a positive integer.")

    return MENU[item] * qty


def check_availability(item: str) -> bool:
    """Check whether an item is listed on the CampusBites menu."""

    return item.lower().strip() in MENU

In [35]:
history = [
    {"role": "system", "content": SYSTEM}
]

MAX_TURNS = 4


def chat(user_text: str) -> str:

    """Generate a customer response using conversation history."""

    user_message = {
        "role": "user",
        "content": user_text
    }

    recent = (
        [history[0]]
        + history[1:][-(MAX_TURNS * 2):]
        + [user_message]
    )

    try:

        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content

    except RateLimitError:
        return "We are a little busy. Please try again shortly."

    except AuthenticationError:
        return "Configuration error. Please contact the administrator."

    except APIConnectionError:
        return "Unable to connect to the server. Please try again."

    if not reply:
        return "Sorry, I could not generate a response."

    history.append(user_message)

    history.append({
        "role": "assistant",
        "content": reply
    })

    return reply

In [36]:
print("Total for 6 dosas:", get_price("dosa", 6))

print("Dosa available:", check_availability("dosa"))

print("Shawarma available:", check_availability("shawarma"))

[LOG] get_price('dosa', 6) -> 210
Total for 6 dosas: 210
Dosa available: True
Shawarma available: False


In [37]:
print("=" * 45)
print("Welcome to CampusBites AI Assistant!")
print("Type 'exit' to end the conversation.")
print("=" * 45)

while True:

    user_text = input("You: ").strip()

    if user_text.lower() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting CampusBites!")
        break

    if not user_text:
        continue

    print("Bot:", chat(user_text))

Welcome to CampusBites AI Assistant!
Type 'exit' to end the conversation.
You: hey
Bot: Hello! 👋 How can I help you with CampusBites today?
You: what's up?
Bot: Hey there! 👋 How can I assist you with CampusBites today?
You: how much shawarma costs?
Bot: I’m sorry, but shawarma isn’t on our menu. For any other items or special requests, please contact the canteen manager.
You: dosa?
Bot: Dosa costs ₹35. Let me know if you’d like to add anything else!
You: exit
Bot: Thank you for visiting CampusBites!
